#### 1. Imports and Bronze tables

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Makes failed to_date attempts return NULL so coalesce can try
# the second date format.
spark.conf.set("spark.sql.ansi.enabled", "false")

bronze_customers = spark.table("bronze_customers")
bronze_rentals = spark.table("bronze_rentals")
bronze_billing = spark.table("bronze_billing")
bronze_depots = spark.table("bronze_depots")

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 3, Finished, Available, Finished, False)

#### 2. Show the damage before cleaning

##### Customer type spellings

In [2]:
display(
    bronze_customers
    .groupBy("CUSTOMER_TYPE")
    .count()
    .orderBy("CUSTOMER_TYPE")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb5dc402-9ed7-484f-8373-1a33ba017d1f)

##### Rental type spellings

In [3]:
display(
    bronze_rentals
    .groupBy("rental_type")
    .count()
    .orderBy("rental_type")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e709e7b7-08d3-4fca-8740-434157641821)

##### Payer type spellings

In [4]:
display(
    bronze_billing
    .groupBy("PAYER_TYPE")
    .count()
    .orderBy("PAYER_TYPE")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b96e159c-a85f-4440-b3b7-3d2ad231dbee)

##### Customer duplicates

In [5]:
customer_duplicate_groups = (
    bronze_customers
    .groupBy("CUSTOMER_ID")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"), "CUSTOMER_ID")
)

display(customer_duplicate_groups)

customer_total = bronze_customers.count()
customer_unique = (
    bronze_customers
    .select("CUSTOMER_ID")
    .distinct()
    .count()
)

print(f"Bronze customer rows:     {customer_total}")
print(f"Distinct customer IDs:    {customer_unique}")
print(f"Duplicate excess rows:    {customer_total - customer_unique}")

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cba3da97-bd4f-422c-b4d4-c14afe6ce670)

Bronze customer rows:     602
Distinct customer IDs:    600
Duplicate excess rows:    2


##### Rental duplicates

In [6]:
rental_duplicate_groups = (
    bronze_rentals
    .groupBy("rental_id")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_file").alias("source_file_count")
    )
    .filter(F.col("row_count") > 1)
    .orderBy("rental_id")
)

display(rental_duplicate_groups)

rental_total = bronze_rentals.count()
rental_unique = (
    bronze_rentals
    .select("rental_id")
    .distinct()
    .count()
)

print(f"Bronze rental rows:       {rental_total}")
print(f"Distinct rental IDs:      {rental_unique}")
print(f"Duplicate excess rows:    {rental_total - rental_unique}")

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3baaa204-2edf-456b-9aa2-f6828f9528f8)

Bronze rental rows:       976
Distinct rental IDs:      945
Duplicate excess rows:    31


#### 3. Clean silver_customers

##### 3.1 Deduplicate by CUSTOMER_ID

In [7]:
customer_window = (
    Window
    .partitionBy("CUSTOMER_ID")
    .orderBy(
        F.col("ingested_at").desc(),
        F.col("source_file").desc()
    )
)

customers_deduped = (
    bronze_customers
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    "Customers after deduplication:",
    customers_deduped.count()
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 9, Finished, Available, Finished, False)

Customers after deduplication: 600


##### 3.2 Demonstrate the single-format date problem

In [8]:
customers_single_date_test = (
    customers_deduped
    .withColumn(
        "_single_format_date",
        F.to_date(
            F.col("REGISTERED_ON"),
            "dd-MM-yyyy"
        )
    )
)

single_format_failures = (
    customers_single_date_test
    .filter(
        (F.col("REGISTERED_ON").isNotNull()) &
        (F.trim(F.col("REGISTERED_ON")) != "") &
        (F.col("_single_format_date").isNull())
    )
    .count()
)

print(
    "A single-format date parse would have "
    f"silently nulled {single_format_failures} of "
    f"{customers_deduped.count()} dates."
)

assert single_format_failures == 273, (
    f"Expected 273 single-format failures, "
    f"found {single_format_failures}"
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 10, Finished, Available, Finished, False)

A single-format date parse would have silently nulled 273 of 600 dates.


##### 3.3 Coalesce both date formats and standardise customer type

In [9]:
customer_type_normalised = F.upper(
    F.regexp_replace(
        F.trim(F.col("CUSTOMER_TYPE")),
        r"[^A-Za-z]",
        ""
    )
)

silver_customers = (
    customers_deduped

    # Clean descriptive columns.
    .withColumn(
        "customer_name",
        F.initcap(F.trim(F.col("CUSTOMER_NAME")))
    )
    .withColumn(
        "city",
        F.initcap(F.trim(F.col("CITY")))
    )

    # Try both REGISTERED_ON formats.
    .withColumn(
        "registered_on",
        F.coalesce(
            F.to_date(
                F.col("REGISTERED_ON"),
                "dd-MM-yyyy"
            ),
            F.to_date(
                F.col("REGISTERED_ON"),
                "yyyy/MM/dd"
            )
        )
    )

    # KYC date has one clean format.
    .withColumn(
        "kyc_verified_on",
        F.to_date(
            F.col("KYC_VERIFIED_ON"),
            "yyyy-MM-dd"
        )
    )

    # Collapse all source spellings to three values.
    .withColumn(
        "customer_type",
        F.when(
            customer_type_normalised.rlike(r"^IND"),
            F.lit("individual")
        )
        .when(
            customer_type_normalised.rlike(
                r"^(CONTR|CONTRACT|CNTR|CTR)"
            ),
            F.lit("contractor")
        )
        .when(
            customer_type_normalised.rlike(
                r"^(COMP|CORP|CO$)"
            ),
            F.lit("company")
        )
    )

    .select(
        F.col("CUSTOMER_ID").alias("customer_id"),
        "customer_name",
        "registered_on",
        "kyc_verified_on",
        "customer_type",
        "city",
        "ingested_at",
        "source_file"
    )
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 11, Finished, Available, Finished, False)

##### 3.4 Assert zero date failures and zero unmapped types

In [10]:
customer_date_failures = (
    silver_customers
    .filter(F.col("registered_on").isNull())
    .count()
)

unmapped_customer_types = (
    silver_customers
    .filter(F.col("customer_type").isNull())
    .count()
)

customer_type_count = (
    silver_customers
    .select("customer_type")
    .distinct()
    .count()
)

print(f"Dual-format date parse failures: {customer_date_failures}")
print(f"Unmapped customer types:         {unmapped_customer_types}")
print(f"Distinct customer types:         {customer_type_count}")

assert customer_date_failures == 0
assert unmapped_customer_types == 0
assert customer_type_count == 3

display(
    silver_customers
    .groupBy("customer_type")
    .count()
    .orderBy("customer_type")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 12, Finished, Available, Finished, False)

Dual-format date parse failures: 0
Unmapped customer types:         0
Distinct customer types:         3


SynapseWidget(Synapse.DataFrame, 7fc1e2be-a69c-46b9-b089-a72cb994526a)

#### 4. Clean silver_rentals

##### 4.1 Deduplicate by rental_id

In [11]:
rental_window = (
    Window
    .partitionBy("rental_id")
    .orderBy(
        F.col("ingested_at").desc(),
        F.col("source_file").desc()
    )
)

rentals_deduped = (
    bronze_rentals
    .withColumn(
        "_row_number",
        F.row_number().over(rental_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

print(
    "Rentals after resend deduplication:",
    rentals_deduped.count()
)

assert rentals_deduped.count() == 945

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 13, Finished, Available, Finished, False)

Rentals after resend deduplication: 945


##### 4.2 Parse timestamps and standardise rental type

In [12]:
rental_type_normalised = F.upper(
    F.regexp_replace(
        F.trim(F.col("rental_type")),
        r"[^A-Za-z]",
        ""
    )
)

rentals_pre_quality = (
    rentals_deduped
    .withColumn(
        "checkout_ts",
        F.to_timestamp(
            F.col("checkout_ts"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "checkin_ts",
        F.to_timestamp(
            F.when(
                F.trim(F.col("checkin_ts")) == "",
                F.lit(None)
            ).otherwise(F.col("checkin_ts")),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "rental_type",
        F.when(
            rental_type_normalised.rlike(
                r"^(STANDARD|STD)"
            ),
            F.lit("standard")
        )
        .when(
            rental_type_normalised.rlike(
                r"^(PRIORITY|PRIO|PRI)"
            ),
            F.lit("priority")
        )
    )
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 14, Finished, Available, Finished, False)

###### Confirm timestamp parsing and type mapping:

In [13]:
checkout_parse_failures = (
    rentals_pre_quality
    .filter(F.col("checkout_ts").isNull())
    .count()
)

unmapped_rental_types = (
    rentals_pre_quality
    .filter(F.col("rental_type").isNull())
    .count()
)

print(f"Checkout timestamp failures: {checkout_parse_failures}")
print(f"Unmapped rental types:        {unmapped_rental_types}")

assert checkout_parse_failures == 0
assert unmapped_rental_types == 0

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 15, Finished, Available, Finished, False)

Checkout timestamp failures: 0
Unmapped rental types:        0


##### 4.3 Apply the quality rule null-safely

In [14]:
bad = (
    F.col("checkin_ts") <
    F.col("checkout_ts")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 16, Finished, Available, Finished, False)

###### Quarantine impossible rentals

In [17]:
silver_rentals_quarantine = (
    rentals_pre_quality
    .filter(
        F.coalesce(
            bad,
            F.lit(False)
        )
    )
    .withColumn(
        "quarantine_reason",
        F.lit("checkin_before_checkout")
    )
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 19, Finished, Available, Finished, False)

###### Show exactly how many rows null safety saved

In [18]:
still_out_count = (
    rentals_pre_quality
    .filter(F.col("checkin_ts").isNull())
    .count()
)

careless_count = careless_rentals.count()
safe_count = silver_rentals.count()
quarantine_count = silver_rentals_quarantine.count()

rows_saved_by_null_safety = (
    safe_count - careless_count
)

print(f"Still-out rentals:                 {still_out_count}")
print(f"Rows kept by careless filter:      {careless_count}")
print(f"Rows kept by null-safe filter:     {safe_count}")
print(f"Rows quarantined:                  {quarantine_count}")
print(
    "Still-out rentals saved by "
    f"null safety: {rows_saved_by_null_safety}"
)

assert still_out_count == 175
assert rows_saved_by_null_safety == 175
assert safe_count == 942
assert quarantine_count == 3

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 20, Finished, Available, Finished, False)

Still-out rentals:                 175
Rows kept by careless filter:      767
Rows kept by null-safe filter:     942
Rows quarantined:                  3
Still-out rentals saved by null safety: 175


###### Validate the final rental types:

In [19]:
rental_type_count = (
    silver_rentals
    .select("rental_type")
    .distinct()
    .count()
)

assert rental_type_count == 2

display(
    silver_rentals
    .groupBy("rental_type")
    .count()
    .orderBy("rental_type")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7fb1e75-7ec0-4f6b-bf47-7bdb1eee5031)

##### 5. Create silver_depots

In [20]:
silver_depots = (
    bronze_depots
    .select(
        F.trim(F.col("DEPOT_CODE")).alias("depot_code"),
        F.trim(F.col("DEPOT_NAME")).alias("depot_name"),
        F.trim(F.col("ZONE")).alias("zone"),
        F.col("FLEET_SIZE").cast("int").alias("fleet_size"),
        "ingested_at",
        "source_file"
    )
)

assert silver_depots.count() == 6

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 22, Finished, Available, Finished, False)

#### 6. Clean billing and parse the amount

##### 6.1 Parse dates, money, and payer type

In [25]:
# Remove the Rs. prefix first, then remove thousands commas.
amount_text_clean = (
    F.regexp_replace(
        F.trim(F.col("AMOUNT_INR")),
        r"(?i)^Rs\.\s*",
        ""
    )
)

amount_text_clean = F.regexp_replace(
    amount_text_clean,
    ",",
    ""
)

billing_clean = (
    bronze_billing
    .withColumn(
        "bill_date",
        F.to_date(
            F.col("BILL_DATE"),
            "yyyy-MM-dd"
        )
    )
    .withColumn(
        "amount_text_clean",
        amount_text_clean
    )
    .withColumn(
        "amount_inr",
        F.col("amount_text_clean").cast("decimal(18,2)")
    )
    .withColumn(
        "payer_type",
        F.lower(F.trim(F.col("PAYER_TYPE")))
    )
    .select(
        F.col("BILL_ID").alias("bill_id"),
        F.col("RENTAL_ID").alias("rental_id"),
        "bill_date",
        "amount_inr",
        "payer_type",
        "ingested_at",
        "source_file"
    )
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 27, Finished, Available, Finished, False)

##### 6.2 Assert zero amount failures

In [26]:
amount_parse_failures = (
    billing_clean
    .filter(F.col("amount_inr").isNull())
    .count()
)

bill_date_failures = (
    billing_clean
    .filter(F.col("bill_date").isNull())
    .count()
)

invalid_payer_types = (
    billing_clean
    .filter(
        ~F.col("payer_type").isin(
            "direct",
            "contract",
            "corporate",
            "prepaid"
        )
    )
    .count()
)

print(f"Amount parse failures: {amount_parse_failures}")
print(f"Bill-date failures:    {bill_date_failures}")
print(f"Invalid payer types:   {invalid_payer_types}")

assert amount_parse_failures == 0
assert bill_date_failures == 0
assert invalid_payer_types == 0

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 28, Finished, Available, Finished, False)

Amount parse failures: 0
Bill-date failures:    0
Invalid payer types:   0


#### 7. Left join billing to rentals

In [27]:
rentals_for_billing = (
    silver_rentals
    .select(
        F.col("rental_id").alias("_matched_rental_id"),
        "customer_id",
        "depot_code",
        "asset_id",
        "checkout_ts",
        "checkin_ts",
        "rental_type"
    )
)

billing_with_rentals = (
    billing_clean.alias("b")
    .join(
        rentals_for_billing.alias("r"),
        F.col("b.rental_id") ==
        F.col("r._matched_rental_id"),
        "left"
    )
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 29, Finished, Available, Finished, False)

##### 7.1 Route unmatched bills to a side table

In [28]:
silver_billing_unmatched = (
    billing_with_rentals
    .filter(F.col("_matched_rental_id").isNull())
    .drop(
        "_matched_rental_id",
        "customer_id",
        "depot_code",
        "asset_id",
        "checkout_ts",
        "checkin_ts",
        "rental_type"
    )
    .withColumn(
        "unmatched_reason",
        F.lit("rental_not_found")
    )
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 30, Finished, Available, Finished, False)

##### 7.2 Keep matched bills in silver_billing

In [29]:
silver_billing = (
    billing_with_rentals
    .filter(F.col("_matched_rental_id").isNotNull())
    .drop("_matched_rental_id")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 31, Finished, Available, Finished, False)

###### Validate the split:

In [30]:
matched_billing_count = silver_billing.count()
unmatched_billing_count = silver_billing_unmatched.count()
total_billing_count = billing_clean.count()

print(f"Matched Silver bills:   {matched_billing_count}")
print(f"Unmatched bills:        {unmatched_billing_count}")
print(f"Total bills preserved:  {total_billing_count}")

assert matched_billing_count == 779
assert unmatched_billing_count == 8
assert matched_billing_count + unmatched_billing_count == 787

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 32, Finished, Available, Finished, False)

Matched Silver bills:   779
Unmatched bills:        8
Total bills preserved:  787


###### Inspect the eight orphans:

In [31]:
display(
    silver_billing_unmatched
    .select(
        "bill_id",
        "rental_id",
        "bill_date",
        "amount_inr",
        "unmatched_reason"
    )
    .orderBy("bill_id")
)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9c5e931-d140-40ea-a12c-03e5f35bb77a)

#### 8. Write all Silver and side tables

In [32]:
def write_delta_table(dataframe, table_name):
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Created: {table_name}")


silver_tables = {
    "silver_customers": silver_customers,
    "silver_rentals": silver_rentals,
    "silver_depots": silver_depots,
    "silver_billing": silver_billing,
    "silver_rentals_quarantine": silver_rentals_quarantine,
    "silver_billing_unmatched": silver_billing_unmatched,
}

for table_name, dataframe in silver_tables.items():
    write_delta_table(dataframe, table_name)

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 34, Finished, Available, Finished, False)

Created: silver_customers
Created: silver_rentals
Created: silver_depots
Created: silver_billing
Created: silver_rentals_quarantine
Created: silver_billing_unmatched


#### 9. Final acceptance validation

In [33]:
expected_counts = {
    "silver_customers": 600,
    "silver_rentals": 942,
    "silver_rentals_quarantine": 3,
    "silver_depots": 6,
    "silver_billing": 779,
    "silver_billing_unmatched": 8,
}

for table_name, expected_count in expected_counts.items():
    actual_count = spark.table(table_name).count()

    print(
        f"{table_name:<30} "
        f"expected={expected_count:<4} "
        f"actual={actual_count:<4}"
    )

    assert actual_count == expected_count, (
        f"{table_name}: expected {expected_count}, "
        f"found {actual_count}"
    )

assert (
    spark.table("silver_customers")
    .select("customer_type")
    .distinct()
    .count()
) == 3

assert (
    spark.table("silver_rentals")
    .select("rental_type")
    .distinct()
    .count()
) == 2

assert customer_date_failures == 0
assert amount_parse_failures == 0
assert single_format_failures == 273
assert rows_saved_by_null_safety == 175

print("\nAll Silver acceptance criteria passed.")

StatementMeta(, 904bf6e1-f391-4ed3-a7e4-042eb9370afa, 35, Finished, Available, Finished, False)

silver_customers               expected=600  actual=600 
silver_rentals                 expected=942  actual=942 
silver_rentals_quarantine      expected=3    actual=3   
silver_depots                  expected=6    actual=6   
silver_billing                 expected=779  actual=779 
silver_billing_unmatched       expected=8    actual=8   

All Silver acceptance criteria passed.
